# AI/ML Cloud Services Cheatsheet

> CLI commands and patterns for AWS SageMaker, Azure Machine Learning, and GCP Vertex AI.

---

## Table of Contents

- [AWS SageMaker](#aws-sagemaker)
- [Azure Machine Learning](#azure-machine-learning)
- [GCP Vertex AI](#gcp-vertex-ai)
- [Cloud Comparison Matrix](#cloud-comparison-matrix)
- [Cost Optimization Tips](#cost-optimization-tips)
- [Interview Scenarios](#interview-scenarios)

---

## AWS SageMaker

### Setup



In [ ]:
# Install SageMaker CLI/SDK
pip install sagemaker boto3

# Configure AWS credentials
aws configure



### Training Jobs



In [ ]:
# Create a training job
aws sagemaker create-training-job \
  --training-job-name "my-training-$(date +%Y%m%d)" \
  --algorithm-specification '{
    "TrainingImage": "763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.0-gpu-py310",
    "TrainingInputMode": "File"
  }' \
  --role-arn "arn:aws:iam::123456789:role/SageMakerRole" \
  --input-data-config '[{
    "ChannelName": "training",
    "DataSource": {
      "S3DataSource": {
        "S3DataType": "S3Prefix",
        "S3Uri": "s3://my-bucket/data/train/",
        "S3DataDistributionType": "FullyReplicated"
      }
    }
  }]' \
  --output-data-config '{
    "S3OutputPath": "s3://my-bucket/output/"
  }' \
  --resource-config '{
    "InstanceCount": 1,
    "InstanceType": "ml.p3.2xlarge",
    "VolumeSizeInGB": 50
  }' \
  --stopping-condition '{"MaxRuntimeInSeconds": 86400}'

# Check training job status
aws sagemaker describe-training-job \
  --training-job-name "my-training-20240101"

# List training jobs
aws sagemaker list-training-jobs \
  --sort-by CreationTime --sort-order Descending --max-results 10



### SageMaker Python SDK (Higher-Level)



In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch

session = sagemaker.Session()
role = sagemaker.get_execution_role()

# Define estimator
estimator = PyTorch(
    entry_point="train.py",
    source_dir="src/",
    role=role,
    instance_count=1,
    instance_type="ml.p3.2xlarge",
    framework_version="2.0",
    py_version="py310",
    hyperparameters={
        "epochs": 10,
        "batch-size": 64,
        "learning-rate": 0.001,
    },
    use_spot_instances=True,            # Save up to 90% cost
    max_wait=7200,                       # Max wait for spot
    max_run=3600,                        # Max training time
    checkpoint_s3_uri="s3://bucket/checkpoints/",
)

# Start training
estimator.fit({
    "training": "s3://my-bucket/data/train/",
    "validation": "s3://my-bucket/data/val/",
})



### Endpoints



In [ ]:
# Create a model
aws sagemaker create-model \
  --model-name "my-model" \
  --primary-container '{
    "Image": "763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.0-gpu-py310",
    "ModelDataUrl": "s3://my-bucket/output/model.tar.gz"
  }' \
  --execution-role-arn "arn:aws:iam::123456789:role/SageMakerRole"

# Create endpoint config
aws sagemaker create-endpoint-config \
  --endpoint-config-name "my-endpoint-config" \
  --production-variants '[{
    "VariantName": "primary",
    "ModelName": "my-model",
    "InstanceType": "ml.g4dn.xlarge",
    "InitialInstanceCount": 1,
    "InitialVariantWeight": 1.0
  }]'

# Create endpoint
aws sagemaker create-endpoint \
  --endpoint-name "my-endpoint" \
  --endpoint-config-name "my-endpoint-config"

# Invoke endpoint
aws sagemaker-runtime invoke-endpoint \
  --endpoint-name "my-endpoint" \
  --content-type "application/json" \
  --body '{"instances": [[1.0, 2.0, 3.0, 4.0]]}' \
  output.json

# Delete endpoint (stop billing)
aws sagemaker delete-endpoint --endpoint-name "my-endpoint"



### SageMaker Processing Jobs (Data Processing)



In [ ]:
aws sagemaker create-processing-job \
  --processing-job-name "data-processing-$(date +%Y%m%d)" \
  --processing-resources '{
    "ClusterConfig": {
      "InstanceCount": 1,
      "InstanceType": "ml.m5.xlarge",
      "VolumeSizeInGB": 30
    }
  }' \
  --app-specification '{
    "ImageUri": "763104351884.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
    "ContainerEntrypoint": ["python3", "/opt/ml/processing/code/preprocess.py"]
  }' \
  --processing-inputs '[{
    "InputName": "raw-data",
    "S3Input": {
      "S3Uri": "s3://bucket/raw/",
      "LocalPath": "/opt/ml/processing/input",
      "S3DataType": "S3Prefix"
    }
  }]' \
  --processing-output-config '{
    "Outputs": [{
      "OutputName": "processed-data",
      "S3Output": {
        "S3Uri": "s3://bucket/processed/",
        "LocalPath": "/opt/ml/processing/output",
        "S3UploadMode": "EndOfJob"
      }
    }]
  }' \
  --role-arn "arn:aws:iam::123456789:role/SageMakerRole"



---

## Azure Machine Learning

### Setup



In [ ]:
# Install Azure ML CLI extension
az extension add -n ml

# Create workspace
az ml workspace create \
  --name my-ml-workspace \
  --resource-group my-rg \
  --location eastus

# Set defaults
az configure --defaults group=my-rg workspace=my-ml-workspace



### Compute



In [ ]:
# Create compute cluster (for training)
az ml compute create \
  --name gpu-cluster \
  --type AmlCompute \
  --size Standard_NC6s_v3 \
  --min-instances 0 \
  --max-instances 4 \
  --idle-time-before-scale-down 300

# Create compute instance (for development)
az ml compute create \
  --name dev-instance \
  --type ComputeInstance \
  --size Standard_DS3_v2

# List compute resources
az ml compute list -o table

# Stop compute instance (save costs)
az ml compute stop --name dev-instance



### Training Jobs



In [ ]:
# job.yml
$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
command: python train.py --epochs ${{inputs.epochs}} --lr ${{inputs.learning_rate}}
environment:
  image: mcr.microsoft.com/azureml/openmpi4.1.0-cuda11.8-cudnn8-ubuntu22.04
  conda_file: conda.yml
compute: azureml:gpu-cluster
inputs:
  epochs: 10
  learning_rate: 0.001
  training_data:
    type: uri_folder
    path: azureml:training-data@latest
code: ./src
experiment_name: my-experiment


In [ ]:
# Submit a training job
az ml job create -f job.yml

# Submit a sweep job (hyperparameter tuning)
az ml job create -f sweep.yml

# Monitor job
az ml job show --name <job-name> -o table

# Stream job logs
az ml job stream --name <job-name>

# Download job outputs
az ml job download --name <job-name> --output-name model

# List recent jobs
az ml job list --max-results 10 -o table

# Cancel a job
az ml job cancel --name <job-name>



### Hyperparameter Sweep



In [ ]:
# sweep.yml
$schema: https://azuremlschemas.azureedge.net/latest/sweepJob.schema.json
type: sweep
trial:
  command: python train.py --lr ${{search_space.lr}} --batch-size ${{search_space.batch_size}}
  environment:
    image: mcr.microsoft.com/azureml/openmpi4.1.0-cuda11.8-cudnn8-ubuntu22.04
  code: ./src
  compute: azureml:gpu-cluster
search_space:
  lr:
    type: loguniform
    min_value: -5
    max_value: -1
  batch_size:
    type: choice
    values: [16, 32, 64, 128]
objective:
  primary_metric: val_accuracy
  goal: maximize
sampling_algorithm: bayesian
limits:
  max_total_trials: 20
  max_concurrent_trials: 4
  timeout: 7200



### Endpoints (Online Inference)



In [ ]:
# endpoint.yml
$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineEndpoint.schema.json
name: fraud-endpoint
auth_mode: key


In [ ]:
# deployment.yml
$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineDeployment.schema.json
name: blue
endpoint_name: fraud-endpoint
model: azureml:fraud-model@latest
code_configuration:
  code: ./src
  scoring_script: score.py
environment:
  image: mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04
  conda_file: conda.yml
instance_type: Standard_DS3_v2
instance_count: 1


In [ ]:
# Create endpoint
az ml online-endpoint create -f endpoint.yml

# Create deployment
az ml online-deployment create -f deployment.yml --all-traffic

# Test endpoint
az ml online-endpoint invoke \
  --name fraud-endpoint \
  --request-file sample-request.json

# Scale deployment
az ml online-deployment update \
  --name blue \
  --endpoint fraud-endpoint \
  --instance-count 3

# Get endpoint scoring URI and key
az ml online-endpoint show --name fraud-endpoint
az ml online-endpoint get-credentials --name fraud-endpoint

# Delete endpoint (stop billing)
az ml online-endpoint delete --name fraud-endpoint --yes



### Data Assets



In [ ]:
# Register a data asset
az ml data create \
  --name training-data \
  --version 1 \
  --type uri_folder \
  --path "https://mystorageaccount.blob.core.windows.net/data/train/"

# List data assets
az ml data list -o table



---

## GCP Vertex AI

### Setup



In [ ]:
# Install Vertex AI SDK
pip install google-cloud-aiplatform

# Authenticate
gcloud auth application-default login

# Set project
gcloud config set project my-project-id



### Training (Custom)



In [ ]:
# Submit a custom training job
gcloud ai custom-jobs create \
  --region=us-central1 \
  --display-name="my-training-job" \
  --worker-pool-spec=machine-type=n1-standard-8,accelerator-type=NVIDIA_TESLA_V100,accelerator-count=1,replica-count=1,container-image-uri=us-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-0:latest \
  --args="--epochs=10,--lr=0.001"

# List training jobs
gcloud ai custom-jobs list --region=us-central1

# Describe a job
gcloud ai custom-jobs describe JOB_ID --region=us-central1



### Vertex AI Python SDK



In [ ]:
from google.cloud import aiplatform

aiplatform.init(project="my-project", location="us-central1")

# Custom training job
job = aiplatform.CustomTrainingJob(
    display_name="my-training",
    script_path="train.py",
    container_uri="us-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-0:latest",
    requirements=["transformers", "datasets"],
)

model = job.run(
    replica_count=1,
    machine_type="n1-standard-8",
    accelerator_type="NVIDIA_TESLA_V100",
    accelerator_count=1,
)



### Endpoints



In [ ]:
# Upload model
gcloud ai models upload \
  --region=us-central1 \
  --display-name="fraud-model" \
  --container-image-uri="us-docker.pkg.dev/vertex-ai/prediction/pytorch-gpu.2-0:latest" \
  --artifact-uri="gs://my-bucket/model/"

# Create endpoint
gcloud ai endpoints create \
  --region=us-central1 \
  --display-name="fraud-endpoint"

# Deploy model to endpoint
gcloud ai endpoints deploy-model ENDPOINT_ID \
  --region=us-central1 \
  --model=MODEL_ID \
  --display-name="fraud-v1" \
  --machine-type=n1-standard-4 \
  --accelerator-type=NVIDIA_TESLA_T4 \
  --accelerator-count=1 \
  --min-replica-count=1 \
  --max-replica-count=3

# Predict
gcloud ai endpoints predict ENDPOINT_ID \
  --region=us-central1 \
  --json-request=request.json

# Undeploy (stop billing)
gcloud ai endpoints undeploy-model ENDPOINT_ID \
  --region=us-central1 \
  --deployed-model-id=DEPLOYED_MODEL_ID



### Vertex AI Pipelines



In [ ]:
from kfp import dsl
from google.cloud import aiplatform

@dsl.pipeline(name="vertex-ml-pipeline")
def ml_pipeline(project: str, region: str):
    from google_cloud_pipeline_components.v1.custom_job import CustomTrainingJobOp
    from google_cloud_pipeline_components.v1.endpoint import (
        EndpointCreateOp,
        ModelDeployOp,
    )

    training_op = CustomTrainingJobOp(
        display_name="train",
        project=project,
        location=region,
        worker_pool_specs=[{
            "machine_spec": {
                "machine_type": "n1-standard-8",
                "accelerator_type": "NVIDIA_TESLA_V100",
                "accelerator_count": 1,
            },
            "replica_count": 1,
            "container_spec": {
                "image_uri": "gcr.io/my-project/training:latest",
            },
        }],
    )

aiplatform.init(project="my-project", location="us-central1")
job = aiplatform.PipelineJob(
    display_name="ml-pipeline",
    template_path="pipeline.json",
)
job.run()



---

## Cloud Comparison Matrix - ML Platform Core

| Feature | AWS SageMaker | Azure ML | GCP Vertex AI |
|---------|--------------|----------|---------------|
| **Managed Notebooks** | SageMaker Studio | Compute Instances | Workbench |
| **Training** | Training Jobs | Command Jobs | Custom Jobs |
| **HPO** | Automatic Model Tuning | Sweep Jobs | Vizier |
| **Model Registry** | Model Registry | Model Registry | Model Registry |
| **Real-time Inference** | Endpoints | Online Endpoints | Endpoints |
| **Batch Inference** | Batch Transform | Batch Endpoints | Batch Prediction |
| **Pipelines** | SageMaker Pipelines | Azure ML Pipelines | Vertex Pipelines |
| **Feature Store** | Feature Store | Managed Feature Store | Feature Store |
| **AutoML** | Autopilot | AutoML | AutoML |
| **GPU Cheapest** | ml.g4dn.xlarge (~$0.53/hr) | Standard_NC4as_T4_v3 (~$0.53/hr) | n1-standard-4 + T4 (~$0.55/hr) |

---

## Complete AI/ML Services Comparison - Every Product Across All Three Clouds

> **The definitive side-by-side comparison of every AI/ML service offered by AWS, Microsoft Azure, and Google Cloud as of 2026.**

---

### 1. Generative AI & Foundation Model Platforms

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Foundation Model Platform** | [Amazon Bedrock](https://aws.amazon.com/bedrock/) | [Azure OpenAI Service](https://azure.microsoft.com/en-us/products/ai-services/openai-service/) | [Gemini Enterprise Agent Platform (Vertex AI)](https://cloud.google.com/vertex-ai) |
| **Model Garden / Catalog** | Bedrock Model Catalog (Anthropic, Meta, Mistral, Cohere, Stability, Amazon Titan) | Azure AI Model Catalog (OpenAI, Meta, Mistral, Cohere, Hugging Face) | Model Garden (200+ models: Gemini, PaLM, Llama, Mistral, Claude) |
| **Proprietary Models** | Amazon Nova (Nova Lite, Nova Pro, Nova Premier, Nova Act, Nova Forge) | GPT-4o, GPT-5.x, o1, o3 (via OpenAI partnership) | Gemini 3.x, Gemini 2.x Flash, Gemma 3 (open) |
| **Model Fine-tuning** | Bedrock Custom Model Training, SageMaker JumpStart | Azure OpenAI Fine-tuning, Azure ML Fine-tuning | Vertex AI Tuning (supervised, RLHF, distillation) |
| **Prompt Engineering** | Bedrock Playground, Bedrock Prompt Flows | Azure AI Studio Prompt Flow | Agent Studio (formerly Vertex AI Studio) |
| **Guardrails / Safety** | Bedrock Guardrails | Azure Content Safety, Azure AI Content Filtering | Responsible AI Toolkit, Vertex AI Safety Filters |
| **RAG (Retrieval Augmented)** | Bedrock Knowledge Bases | Azure AI Search + OpenAI On Your Data | Vertex AI RAG Engine, Grounding with Google Search |

**Key Differences:**
- **AWS Bedrock** is the most model-agnostic - offers Anthropic Claude, Meta Llama, Mistral, Cohere, Stability, and their own Titan/Nova models all behind one API.
- **Azure OpenAI** has the exclusive managed deployment of OpenAI models (GPT-4o, GPT-5, o1) with enterprise SLAs.
- **Google** offers the deepest Gemini integration with native multimodal capabilities and the largest open model garden (200+).

---

### 2. Agentic AI & Agent Platforms

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Agent Builder** | [Bedrock Agents](https://aws.amazon.com/bedrock/agents/) | [Foundry Agent Service](https://azure.microsoft.com/en-us/products/ai-services/ai-agent-service/) | [Agent Development Kit (ADK)](https://google.github.io/adk-docs/) |
| **Agent Orchestration** | Bedrock AgentCore (plan, orchestrate, evaluate agents at scale) | Foundry Control Plane (observe, control, secure, govern agents) | Agent Space (discover, create, share, run agents) |
| **Agent Registry** | AWS Agent Registry (preview) | Foundry Agent Registry | Agent Garden (prebuilt sample agents) |
| **Agent Evaluation** | Bedrock AgentCore Evaluations | Foundry Observability (tracing, evaluation) | Vertex AI Evaluation Service |
| **Multi-Agent Systems** | Bedrock Multi-Agent Collaboration | AutoGen, Semantic Kernel Multi-Agent | Agent Development Kit Multi-Agent |
| **Browser Agents** | Amazon Nova Act (UI workflow automation) | - | - |
| **Pre-built Agents** | AWS DevOps Agent, AWS Security Agent, Amazon Quick | Microsoft 365 Copilot Agents, Dynamics 365 Agents | Customer Engagement Suite Agents, Agent Garden templates |

**Key Differences:**
- **AWS** leads with Amazon Bedrock AgentCore - a full lifecycle agent platform with quality evaluations, policy controls, and registry.
- **Azure** has the deepest enterprise integration with Foundry (formerly Azure AI), connecting agents to Microsoft 365, Dynamics, and Teams.
- **Google** has the most open approach with Agent Development Kit (ADK) as an open-source framework and ADK deployed on Cloud Run or GKE.

---

### 3. ML Platform & MLOps

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **End-to-End ML Platform** | [Amazon SageMaker AI](https://aws.amazon.com/sagemaker/) | [Azure Machine Learning](https://azure.microsoft.com/en-us/products/machine-learning/) | [Vertex AI](https://cloud.google.com/vertex-ai) |
| **Managed Notebooks** | SageMaker Studio (JupyterLab) | Compute Instances (JupyterLab) | Vertex AI Workbench, Colab Enterprise |
| **Data Labeling** | SageMaker Ground Truth | Azure ML Data Labeling | Vertex AI Data Labeling |
| **Feature Store** | SageMaker Feature Store | Azure ML Managed Feature Store | Vertex AI Feature Store |
| **Experiment Tracking** | SageMaker Experiments | Azure ML Experiments + MLflow | Vertex AI Experiments + TensorBoard |
| **Model Registry** | SageMaker Model Registry | Azure ML Model Registry | Vertex AI Model Registry |
| **Pipelines / Workflows** | SageMaker Pipelines | Azure ML Pipelines (v2 SDK, Designer) | Vertex AI Pipelines (Kubeflow-based) |
| **Monitoring** | SageMaker Model Monitor | Azure ML Model Monitoring | Vertex AI Model Monitoring |
| **AutoML** | SageMaker Autopilot | Azure AutoML | Vertex AI AutoML (Vision, Tabular, Text, Video, Forecasting) |
| **Hyperparameter Tuning** | Automatic Model Tuning | Sweep Jobs (Bayesian, Grid, Random) | Vertex AI Vizier |
| **Distributed Training** | SageMaker Distributed Training (data/model parallel) | Azure ML Distributed Training (DeepSpeed, Horovod) | Vertex AI Distributed Training (Reduction Server) |
| **MLflow Integration** | SageMaker with MLflow | Native MLflow (first-class) | Vertex AI with MLflow |
| **Managed Spark** | SageMaker Processing (Spark) | Azure Synapse / Databricks | Dataproc Serverless |
| **Low-code / No-code** | SageMaker Canvas | Azure ML Designer (drag-and-drop) | Vertex AI AutoML (no-code console) |

**Key Differences:**
- **Azure ML** has the best MLflow integration (first-class native support) and the most mature designer UI for no-code ML.
- **SageMaker** has the broadest instance selection and the most mature spot-instance training story (up to 90% savings).
- **Vertex AI** has the tightest BigQuery integration and the most advanced AutoML covering vision, text, tabular, video, and forecasting in one service.

---

### 4. Computer Vision

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Pre-trained Vision API** | [Amazon Rekognition](https://aws.amazon.com/rekognition/) | [Azure Vision](https://azure.microsoft.com/en-us/products/ai-services/ai-vision/) | [Cloud Vision AI](https://cloud.google.com/vision) |
| **Object Detection** | Rekognition DetectLabels | Azure Vision Object Detection | Vision AI Object Localization |
| **Face Detection/Analysis** | Rekognition DetectFaces | Azure Face API | Vision AI Face Detection |
| **OCR / Text in Images** | Rekognition DetectText, Textract | Azure Vision OCR, Document Intelligence | Vision AI Text Detection, Document AI |
| **Image Moderation** | Rekognition Content Moderation | Azure Content Moderator / Content Safety | Cloud Vision SafeSearch |
| **Custom Image Classification** | Rekognition Custom Labels | Azure Custom Vision | Vertex AI AutoML Vision |
| **Video Analysis** | Rekognition Video | Azure Video Indexer | Video Intelligence AI |
| **Image Generation** | Amazon Titan Image Generator (Bedrock) | Azure OpenAI DALL-E 3 | Imagen (via Vertex AI) |
| **Satellite / Geospatial** | SageMaker Geospatial | Azure Orbital Analytics | Earth Engine AI Platform |

---

### 5. Natural Language Processing (NLP)

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Text Analytics** | [Amazon Comprehend](https://aws.amazon.com/comprehend/) | [Azure Language](https://azure.microsoft.com/en-us/products/ai-services/ai-language/) | [Natural Language AI](https://cloud.google.com/natural-language) |
| **Sentiment Analysis** | Comprehend Sentiment | Language Sentiment Analysis | NL API Sentiment Analysis |
| **Entity Recognition** | Comprehend Entity Recognition | Language Named Entity Recognition | NL API Entity Analysis |
| **Key Phrase Extraction** | Comprehend Key Phrases | Language Key Phrase Extraction | NL API Content Classification |
| **Language Detection** | Comprehend Dominant Language | Language Language Detection | NL API Language Detection |
| **Custom Text Classification** | Comprehend Custom Classification | Language Custom Text Classification | Vertex AI AutoML Text |
| **Custom NER** | Comprehend Custom Entity Recognition | Language Custom NER | Vertex AI AutoML Entity Extraction |
| **PII Detection / Redaction** | Comprehend PII Detection | Language PII Detection | Cloud DLP (Data Loss Prevention) |
| **Topic Modeling** | Comprehend Topics | - | - |
| **Medical NLP** | [Amazon Comprehend Medical](https://aws.amazon.com/comprehend/medical/) | [Azure Health Text Analytics](https://learn.microsoft.com/en-us/azure/ai-services/language-service/text-analytics-for-health/overview) | [Healthcare Natural Language AI](https://cloud.google.com/healthcare-api/docs/concepts/nlp) |

---

### 6. Speech & Audio

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Speech-to-Text** | [Amazon Transcribe](https://aws.amazon.com/transcribe/) | [Azure Speech-to-Text](https://azure.microsoft.com/en-us/products/ai-services/speech-to-text/) | [Cloud Speech-to-Text](https://cloud.google.com/speech-to-text) |
| **Text-to-Speech** | [Amazon Polly](https://aws.amazon.com/polly/) | [Azure Text-to-Speech](https://azure.microsoft.com/en-us/products/ai-services/text-to-speech/) | [Cloud Text-to-Speech](https://cloud.google.com/text-to-speech) |
| **Real-time Transcription** | Transcribe Streaming | Speech-to-Text Real-time | Speech-to-Text Streaming |
| **Speaker Diarization** | Transcribe Speaker ID | Speech Speaker Diarization | Speech-to-Text Diarization |
| **Custom Vocabulary** | Transcribe Custom Vocabulary | Speech Custom Speech Models | Speech-to-Text Adaptation |
| **Medical Transcription** | Amazon Transcribe Medical | Azure Speech for Healthcare | - |
| **Call Analytics** | Amazon Transcribe Call Analytics | Azure Speech Call Center | CCAI Insights |
| **Voice Cloning** | - | Azure Custom Neural Voice | - |
| **Subtitle Generation** | Transcribe (VTT/SRT output) | Speech Batch Transcription | Speech-to-Text (SRT output) |

---

### 7. Translation & Localization

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Machine Translation** | [Amazon Translate](https://aws.amazon.com/translate/) | [Azure Translator](https://azure.microsoft.com/en-us/products/ai-services/ai-translator/) | [Cloud Translation AI](https://cloud.google.com/translate) |
| **Custom Translation** | Translate Custom Terminology | Translator Custom Translator | Translation AI AutoML |
| **Document Translation** | Translate Batch Translation | Translator Document Translation | Translation AI Batch |
| **Languages Supported** | 75+ languages | 100+ languages | 130+ languages |
| **Real-time Translation** | Translate Real-time API | Translator Real-time | Translation API Real-time |

---

### 8. Document Processing & OCR

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Document Processing** | [Amazon Textract](https://aws.amazon.com/textract/) | [Azure Document Intelligence](https://azure.microsoft.com/en-us/products/ai-services/ai-document-intelligence/) (formerly Form Recognizer) | [Document AI](https://cloud.google.com/document-ai) |
| **OCR** | Textract DetectDocumentText | Document Intelligence Read API | Document AI OCR |
| **Form Extraction** | Textract AnalyzeDocument (Forms) | Document Intelligence Custom Models | Document AI Custom Extractors |
| **Table Extraction** | Textract Tables | Document Intelligence Tables | Document AI Table Parsing |
| **Invoice / Receipt** | Textract AnalyzeExpense | Document Intelligence Prebuilt (Invoice, Receipt) | Document AI Invoice Parser, Receipt Parser |
| **ID Document** | Textract AnalyzeID | Document Intelligence Prebuilt ID | Document AI Identity Processor |
| **Medical Document** | Textract + Comprehend Medical | Health Text Analytics | Healthcare Document AI |
| **Lending / Mortgage** | Textract Lending | - | Document AI Lending |

---

### 9. Conversational AI & Chatbots

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Chatbot Builder** | [Amazon Lex](https://aws.amazon.com/lex/) | [Azure Bot Services](https://azure.microsoft.com/en-us/products/bot-services/) | [CX Agent Studio (Dialogflow CX)](https://cloud.google.com/dialogflow) |
| **Intent-based Bots** | Lex V2 (intents, slots, fulfillment) | Bot Framework Composer | Dialogflow CX Flows |
| **Generative AI Bots** | Lex + Bedrock Knowledge Bases | Azure Bot + OpenAI integration | Dialogflow CX + Generative AI Agents |
| **Voice Bots / IVR** | Amazon Connect + Lex | Azure Bot + Speech | CCAI Virtual Agent + Telephony |
| **Contact Center AI** | [Amazon Connect](https://aws.amazon.com/connect/) | [Dynamics 365 Contact Center](https://learn.microsoft.com/en-us/dynamics365/contact-center/) | [Contact Center AI (CCAI)](https://cloud.google.com/solutions/contact-center) |
| **Agent Assist** | Connect Wisdom (real-time agent assist) | Dynamics 365 Copilot for Service | CCAI Agent Assist |
| **Agent Insights** | Connect Contact Lens (analytics) | Dynamics 365 Omnichannel Analytics | CCAI Insights |

---

### 10. Search & Knowledge

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **AI-Powered Search** | [Amazon Kendra](https://aws.amazon.com/kendra/) | [Azure AI Search](https://azure.microsoft.com/en-us/products/ai-services/ai-search/) (formerly Cognitive Search) | [Agent Search (Vertex AI Search)](https://cloud.google.com/enterprise-search) |
| **Semantic / Vector Search** | Kendra + OpenSearch Serverless | AI Search (vector search, semantic ranking) | Agent Search + AlloyDB / BigQuery vector search |
| **Enterprise Knowledge Base** | Bedrock Knowledge Bases | Azure AI Search + OpenAI On Your Data | Vertex AI RAG Engine, Foundry IQ |
| **E-commerce Search** | - | - | Vertex AI Search for Retail |
| **Recommendation Engine** | [Amazon Personalize](https://aws.amazon.com/personalize/) | Azure Personalizer (retiring) | [Vertex AI Recommendations](https://cloud.google.com/recommendations) |

---

### 11. Forecasting & Time Series

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Managed Forecasting** | [Amazon Forecast](https://aws.amazon.com/forecast/) (legacy) | Azure AutoML Forecasting | Vertex AI AutoML Forecasting |
| **Anomaly Detection** | [Amazon Lookout for Metrics](https://aws.amazon.com/lookout-for-metrics/) | [Azure AI Anomaly Detector](https://azure.microsoft.com/en-us/products/ai-services/ai-anomaly-detector/) | Vertex AI Tabular Workflows |
| **Time Series Insights** | CloudWatch Anomaly Detection | Azure Time Series Insights | Timeseries Insights API |
| **Demand Forecasting** | Forecast + SageMaker | Azure AutoML + Power BI | Vertex AI Forecasting + BigQuery ML |

---

### 12. Data & Business Intelligence AI

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **SQL-based ML** | - | - | [BigQuery ML](https://cloud.google.com/bigquery/docs/bqml-introduction) (CREATE MODEL in SQL) |
| **Business Intelligence AI** | Amazon QuickSight Q (NL queries) | Power BI Copilot (NL queries) | Looker + Gemini (NL queries) |
| **Data Preparation** | SageMaker Data Wrangler | Azure ML Data Prep | Vertex AI Feature Store, Dataprep |
| **ETL with AI** | AWS Glue + ML transforms | Azure Data Factory + ML | Dataflow ML, Dataproc + Spark ML |
| **Streaming ML** | Kinesis Data Analytics ML | Stream Analytics Anomaly Detection | Dataflow ML (streaming) |

---

### 13. AI Infrastructure & Compute

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **GPU Instances** | P5 (H100), P4d (A100), G5 (A10G), G4dn (T4) | NCads A100, NCasT4, NDs H100 v5 | A3 (H100), A2 (A100), G2 (L4) |
| **Custom AI Chips** | AWS Trainium (Trn1, Trn2) - training; AWS Inferentia (Inf2) - inference | Azure Maia 100 (preview) | TPU v5p (training), TPU v5e (inference), TPU v6e |
| **Distributed Training Infra** | SageMaker HyperPod, UltraCluster | Azure ML Distributed Training, ND H100 v5 | Vertex AI Distributed Training, GKE + TPU Multislice |
| **Serverless Inference** | SageMaker Serverless Inference | Azure ML Serverless Endpoints | Cloud Run GPU, Vertex AI Serverless Endpoints |
| **Edge AI** | SageMaker Edge Manager, AWS IoT Greengrass ML | Azure Percept (deprecated), ONNX Runtime | Vertex AI Edge Manager, Coral Edge TPU |
| **Spot / Preemptible** | Spot Instances (up to 90% off) | Low-priority VMs (up to 80% off) | Spot VMs (up to 91% off) |

---

### 14. AI Code Assistants & Developer Tools

| Category | AWS | Azure / Microsoft | Google Cloud |
|----------|-----|-------------------|-------------|
| **Code Assistant** | Amazon Q Developer (formerly CodeWhisperer) | [GitHub Copilot](https://github.com/features/copilot) | [Gemini Code Assist](https://cloud.google.com/gemini/docs/codeassist/overview) |
| **Code Generation** | Q Developer code suggestions | Copilot inline completions + agents | Code Assist completions + chat |
| **Code Review** | Q Developer code review | Copilot code review (PR reviews) | Code Assist code review |
| **CLI Assistant** | Q Developer CLI | GitHub Copilot CLI | gcloud CLI + Gemini |
| **IDE Support** | VS Code, JetBrains, CLI | VS Code, JetBrains, Neovim, Xcode | VS Code, JetBrains, Cloud Shell Editor |
| **Agentic Coding IDE** | Amazon Q Developer Agent | [GitHub Copilot Agent Mode](https://code.visualstudio.com/docs/copilot/overview) | [Google Antigravity](https://antigravity.google/) |
| **Security Scanning** | Q Developer Security Scans | Copilot + GitHub Advanced Security | Code Assist Security Analysis |
| **Pricing (Free Tier)** | Free tier (limited) | Copilot Free (limited monthly) | Free tier (limited) |

---

### 15. Responsible AI & Governance

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Content Safety** | Bedrock Guardrails | Azure Content Safety | Responsible AI Toolkit |
| **Model Explainability** | SageMaker Clarify | Azure ML Responsible AI Dashboard | Vertex AI Explainable AI |
| **Bias Detection** | SageMaker Clarify Bias Detection | Responsible AI Toolbox (Fairlearn) | Vertex AI Fairness Indicators |
| **Model Cards** | SageMaker Model Cards | Azure ML Model Cards | Vertex AI Model Cards |
| **Data Privacy** | Macie (PII), Comprehend PII | Azure Purview, Language PII Detection | Cloud DLP (Data Loss Prevention) |
| **Compliance** | FedRAMP, HIPAA, SOC, PCI | FedRAMP, HIPAA, SOC, PCI, Azure Government | FedRAMP, HIPAA, SOC, PCI, Assured Workloads |

---

### 16. Industry-Specific AI

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Healthcare AI** | Amazon HealthLake, Comprehend Medical, Transcribe Medical | Azure Health Bot, Health Text Analytics, Azure Health Data Services | Healthcare Natural Language AI, Healthcare API, Medical Imaging |
| **Financial Services AI** | Amazon Fraud Detector, SageMaker for Financial Services | Azure Anomaly Detector, Dynamics 365 Fraud Protection | Vertex AI for Financial Services, AML AI |
| **Retail AI** | Amazon Personalize, Forecast | Azure Personalizer, Dynamics 365 Commerce | Vertex AI Search for Retail, Recommendations AI |
| **Manufacturing / Industrial** | Amazon Lookout for Equipment, Lookout for Vision, Monitron | Azure Digital Twins + ML | Vertex AI for Manufacturing, Visual Inspection AI |
| **Media & Entertainment** | Amazon Rekognition, Transcribe, Polly | Azure Video Indexer, Azure Media Services | Video Intelligence AI, Speech-to-Text |
| **Automotive** | AWS IoT FleetWise + ML | Azure Connected Vehicle Platform | Automotive AI (Waymo partnership) |
| **Sustainability** | - | Azure Emissions Impact Dashboard | Carbon Footprint, Earth Engine |

---

### 17. AI Marketplace & Pre-trained Models

| Category | AWS | Azure | Google Cloud |
|----------|-----|-------|-------------|
| **Model Marketplace** | [SageMaker JumpStart](https://aws.amazon.com/sagemaker/jumpstart/) (350+ models) | [Azure AI Model Catalog](https://ai.azure.com/explore/models) (1600+ models) | [Model Garden](https://cloud.google.com/model-garden) (200+ models) |
| **Hugging Face Integration** | SageMaker Hugging Face DLC | Azure ML Hugging Face Hub | Vertex AI Hugging Face Endpoints |
| **Pre-trained Solution Templates** | SageMaker JumpStart Solutions | Azure AI Templates | Agent Garden (prebuilt agent templates) |
| **Marketplace** | AWS Marketplace (ML) | Azure Marketplace (AI) | Google Cloud Marketplace (AI) |

---

### 18. Quick Decision Guide

| If You Need... | Choose | Why |
|----------------|--------|-----|
| Best OpenAI model access (GPT-4o, o1, GPT-5) | **Azure** | Exclusive managed enterprise deployment of OpenAI models |
| Most model choices behind one API | **AWS Bedrock** | Widest foundation model selection (Anthropic, Meta, Mistral, Cohere, Amazon) |
| Best multimodal AI (native vision + text + code) | **Google** | Gemini models are natively multimodal; tightest integration with Search |
| Deepest SQL-based ML | **Google** | BigQuery ML lets you CREATE MODEL with SQL directly |
| Most mature MLOps with MLflow | **Azure** | First-class native MLflow tracking, registry, deployment |
| Cheapest training (spot) | **AWS** | Up to 90% savings with SageMaker managed spot training |
| Best custom silicon | **Google** (TPU) / **AWS** (Trainium/Inferentia) | Google TPUs excel at transformer training; Trainium for price-performance |
| Enterprise SSO / Teams / Office | **Azure** | Deepest Microsoft 365, Dynamics 365, Teams integration |
| Existing Google Workspace shop | **Google** | Native Gemini integration across Workspace apps |
| Existing AWS ecosystem | **AWS** | Tightest S3, Lambda, Step Functions, IAM integration |

---

### Quick Reference: Service Name Mapping

| Capability | AWS | Azure | Google Cloud |
|-----------|-----|-------|-------------|
| Vision API | Rekognition | Azure Vision | Cloud Vision AI |
| Text Analytics | Comprehend | Azure Language | Natural Language AI |
| Speech-to-Text | Transcribe | Azure Speech | Speech-to-Text |
| Text-to-Speech | Polly | Azure Speech | Text-to-Speech |
| Translation | Translate | Azure Translator | Cloud Translation |
| Document OCR | Textract | Document Intelligence | Document AI |
| Chatbot | Lex | Bot Services | Dialogflow CX |
| Search | Kendra | AI Search | Agent Search |
| Recommendations | Personalize | Personalizer | Recommendations AI |
| Forecasting | Forecast | AutoML Forecasting | AutoML Forecasting |
| Anomaly Detection | Lookout for Metrics | AI Anomaly Detector | Vertex AI Tabular |
| Fraud Detection | Fraud Detector | Dynamics 365 Fraud Protection | AML AI |
| Code Assistant | Q Developer | GitHub Copilot | Gemini Code Assist |
| Contact Center | Connect | Dynamics 365 Contact Center | CCAI |
| ML Platform | SageMaker | Azure ML | Vertex AI |
| GenAI Platform | Bedrock | Azure OpenAI | Gemini Enterprise Agent Platform |
| Agent Platform | Bedrock AgentCore | Foundry Agent Service | Agent Development Kit |
| Agentic IDE | Q Developer Agent | GitHub Copilot Agent Mode | Google Antigravity |

---

## Cost Optimization Tips



In [ ]:
# AWS: Use spot instances (up to 90% savings)
# Set use_spot_instances=True in SageMaker estimator

# Azure: Use low-priority VMs
az ml compute create --name gpu-cluster --type AmlCompute \
  --size Standard_NC6s_v3 --tier low_priority

# GCP: Use preemptible VMs
gcloud ai custom-jobs create \
  --worker-pool-spec=machine-type=n1-standard-8,accelerator-type=NVIDIA_TESLA_V100,accelerator-count=1 \
  --enable-web-access

# All clouds: Auto-scale to zero when idle
# Set min-instances 0 for compute clusters

# All clouds: Right-size your instances
# Don't use A100 for fine-tuning small models
# Use T4 for inference, V100/A100 for training



---

## Interview Scenarios

**Q: How would you choose between AWS SageMaker, Azure ML, and GCP Vertex AI?**
> Consider: (1) existing cloud investment and team expertise, (2) specific features needed (e.g., Azure ML has best MLflow integration, SageMaker has broadest instance selection, Vertex AI has best AutoML), (3) pricing for your workload pattern, (4) compliance requirements, (5) integration with other services you use.

**Q: How do you manage costs for ML workloads in the cloud?**
> Key strategies: spot/preemptible instances for training (with checkpointing), auto-scale to zero for compute clusters, right-size GPU selection, reserved instances for steady-state inference, batch inference instead of real-time where possible, and use cloud cost monitoring tools (AWS Cost Explorer, Azure Cost Management, GCP Billing).

**Q: Compare the GenAI platforms - Bedrock vs Azure OpenAI vs Vertex AI?**
> **AWS Bedrock** is the most model-agnostic (Anthropic, Meta, Mistral, Cohere, Amazon Nova behind one unified API). **Azure OpenAI** provides exclusive managed GPT-4o/GPT-5/o1 models with enterprise SLAs and Microsoft ecosystem integration. **Google Vertex AI** has native Gemini (best multimodal), the largest model garden (200+), BigQuery ML (SQL-based ML), and TPU access for cost-effective training.

**Q: When would you choose custom silicon (TPUs, Trainium, Inferentia) over GPUs?**
> TPUs excel at large-scale transformer training with high throughput. Trainium (AWS) offers best price-performance for training. Inferentia (AWS) excels at inference cost. Use GPUs (NVIDIA H100/A100) when you need CUDA ecosystem compatibility, custom CUDA kernels, or broad framework support. Use custom silicon when workload fits and you want 40-60% cost savings over equivalent GPUs.

**Q: How do agentic AI platforms differ across the three clouds?**
> **AWS Bedrock AgentCore** focuses on production-grade agent lifecycle management with quality evaluations, policy controls, and Agent Registry. **Azure Foundry Agent Service** provides the tightest enterprise integration (Teams, M365, Dynamics). **Google ADK** is the most open-source approach, deployed flexibly on Cloud Run or GKE with Agent Garden templates.

---

## Further Reading & Official Documentation

| Provider | Resource | Link |
|----------|----------|------|
| **AWS** | All AI/ML Services | [aws.amazon.com/ai](https://aws.amazon.com/ai/) |
| **AWS** | SageMaker Documentation | [docs.aws.amazon.com/sagemaker](https://docs.aws.amazon.com/sagemaker/) |
| **AWS** | Amazon Bedrock | [aws.amazon.com/bedrock](https://aws.amazon.com/bedrock/) |
| **Azure** | All AI Services | [azure.microsoft.com/products/category/ai](https://azure.microsoft.com/en-us/products/category/ai) |
| **Azure** | Azure Machine Learning | [learn.microsoft.com/azure/machine-learning](https://learn.microsoft.com/en-us/azure/machine-learning/) |
| **Azure** | Azure OpenAI Service | [learn.microsoft.com/azure/ai-services/openai](https://learn.microsoft.com/en-us/azure/ai-services/openai/) |
| **Google** | All AI Products | [cloud.google.com/products/ai](https://cloud.google.com/products/ai) |
| **Google** | Vertex AI Documentation | [cloud.google.com/vertex-ai/docs](https://cloud.google.com/vertex-ai/docs) |
| **Google** | Model Garden | [cloud.google.com/model-garden](https://cloud.google.com/model-garden) |
